In [77]:
import torch
import torch.nn as nn
x = torch.rand([2,3])
x_example = torch.tensor([12,54,10,71])

print(f"Tensor :{x} \n") #tensor([[0.9652, 0.5236, 0.2486],
                               # [0.1699, 0.6405, 0.6060]]) 


 






print(f"Size :{x.shape} \n") # Size :torch.Size([2, 3]) 
print(f"type :{x.dtype} \n") # type :torch.float32
print(f"device :{x.device} \n") # device :cpu 
#x.to("cuda") to change device


Tensor :tensor([[0.9477, 0.6128, 0.6480],
        [0.7900, 0.9741, 0.2097]]) 

Size :torch.Size([2, 3]) 

type :torch.float32 

device :cpu 



In [78]:
criterion = nn.MSELoss()

target = torch.tensor([[10.0, 8.0 , 6.7 , 34.0],[10.0, 8.0 , 6.7 , 34.0]])

In [79]:
layer1 = nn.Linear(3,2)

print(f" layer1's weight :{layer1.weight} \n")
#layer1's weight :Parameter containing:
# tensor([[ 0.1439, -0.0174, -0.3733],
#         [ 0.1121,  0.2467,  0.0511]], requires_grad=True) 


print(f" layer1's bias :{layer1.bias} \n") 
#  layer1's bias :Parameter containing:
# tensor([-0.3743,  0.0898], requires_grad=True)


 layer1's weight :Parameter containing:
tensor([[ 0.1633,  0.1252,  0.0470],
        [ 0.3243, -0.1418,  0.5663]], requires_grad=True) 

 layer1's bias :Parameter containing:
tensor([-0.1133, -0.5550], requires_grad=True) 



In [80]:
class My_Module(nn.Module): # Module + Layers 
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer1 = nn.Linear(3,16) #input layer
        self.layer2 = nn.Linear(16,8) #hidden layer
        self.output = nn.Linear(8,2) # output layer
        self.relu = nn.ReLU()

    
    def forward(self,x):
        x = self.layer1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.relu(x)

        x = self.output(x)
        return x
        #return self.relu(x)

model = My_Module()

y= model(x)
y
#15620033
#x_example.size()

tensor([[0.0374, 0.2004],
        [0.0550, 0.2134]], grad_fn=<AddmmBackward0>)

In [81]:
class My_Module(nn.Module): # Module + Layers + activation function(relu)
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer1 = nn.Linear(3,16) #input layer
        self.layer2 = nn.Linear(16,8) #hidden layer
        self.output = nn.Linear(8,2) # output layer
        self.relu = nn.ReLU()

    
    def forward(self,x):
        x = self.layer1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.relu(x)

        x = self.output(x)
        
        return self.relu(x)

model = My_Module()

y= model(x)
y
#15620033
#x_example.size()

tensor([[0.0000, 0.1401],
        [0.0000, 0.1369]], grad_fn=<ReluBackward0>)

In [82]:
class My_Module(nn.Module): # Module + Layers + activation function(tanh)
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer1 = nn.Linear(3,16) #input layer
        self.layer2 = nn.Linear(16,8) #hidden layer
        self.output = nn.Linear(8,2) # output layer
        #self.activation = nn.ReLU()
        self.activation = nn.Tanh()

    
    def forward(self,x):
        x = self.layer1(x)
        x = self.activation(x)

        x = self.layer2(x)
        x = self.activation(x)

        x = self.output(x)
        
        return self.activation(x)

model = My_Module()

y= model(x)
y
#15620033
#x_example.size()

tensor([[-0.2564, -0.5840],
        [-0.2546, -0.5807]], grad_fn=<TanhBackward0>)

In [83]:
class My_Module(nn.Module): # Module + Layers + activation function(relu) + sequential
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # we replace all of that through the self.network as sequence of linear -> activation  
        
        self.network = nn.Sequential(
            nn.Linear(3,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,4)
        )

    
    def forward(self,x):
        return self.network(x)

model = My_Module()

y= model(x)
y

#15620033
#x_example.size()

tensor([[-0.0522, -0.0609,  0.0613, -0.1410],
        [-0.0316, -0.0684,  0.0539, -0.1016]], grad_fn=<AddmmBackward0>)

In [84]:

prediction = model(x)

loss = criterion(prediction,target)
loss

tensor(343.5661, grad_fn=<MseLossBackward0>)

In [85]:
# I would build my own module as well as a custom loss function
class My_Module(nn.Module):
    def __init__(self, input_dim,hidden_dim,output_dim):
        super().__init__()

        
        self.network = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,output_dim), 
            #nn.ReLU(), # -- no activation , loss decides.
        )
    def forward(self,x):
        return self.network(x)
   

class My_Loss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, prediction , target):
        #write any math here . 1- use only torch operations 2-return a single scalar tensor 

        loss = (prediction-target)**2
        return loss.mean()
    
model = My_Module(3,64,1)
loss_fn = My_Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- data ---
x = torch.randn(32, 3)
y = torch.randn(32)

# --- training loop ---
for epoch in range(100):
    y_pred = model(x)
    loss   = loss_fn(y_pred.squeeze(), y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} loss {loss.item():.4f}")

epoch   0 loss 1.0925


epoch  20 loss 0.8517
epoch  40 loss 0.7148
epoch  60 loss 0.5964
epoch  80 loss 0.5057


# What you define vs what you inherit


In [86]:
class Anything(nn.Module):

    def __init__(self):
        super().__init__()          # ← boots up all the inherited machinery
        self.layer = nn.Linear(4,2) # ← any nn.Module stored as attribute
                                    #   gets tracked automatically

    def forward(self, x):           # ← the ONE thing you must define
        return self.layer(x)        # ← your logic goes here, nothing else

In [90]:
# 3 things , I can build with nn.module
## 1. a layer — computes something, has weights
class MyLinear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(in_dim, out_dim))  # a raw weight
        self.b = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x):
        return x @ self.W + self.b


## 2. a network — stacks layers into a full model
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = MyLinear(3, 64)
        self.l2 = MyLinear(64, 4)

    def forward(self, x):
        return self.l2(F.relu(self.l1(x)))


## 3. a loss function — computes a scalar from prediction + truth
class MyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()

In [91]:
# build freely and flexibly nested

import torch.nn.functional as F

class Block(nn.Module): 
    """a reusable chunk"""
    def __init__(self,dim):
        super().__init__()
        self.fc = nn.Linear(dim,dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self,x):
        return self.norm(F.relu(self.fc(x)))
    

class BigNetwork(nn.Module):
    """built from blocks"""
    def __init__(self):
        super().__init__()
        self.block1 = Block(64)
        self.block2 = Block(64)
        self.head = nn.Linear(64,4)
    def forward(self,x):
        x = self.block1(x)
        x = self.block2(x)
        return self.head(x)
    
net = BigNetwork()
print(len(list(net.parameters())))


10


## How to use loss model


In [ ]:
loss_fn = MyLoss()

y_pred = model(x)
y_target = ...

loss = loss_fn(y_pred, y_target) #loss = loss_fn(value_pred, return_target)

loss.backward()
optimizer.step()